# EDA — Credit Default Prediction & Portfolio Risk

Quick exploratory pass over the raw UCI "Default of Credit Card Clients" data
before any feature engineering. Run `python src/download_data.py` first.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import plotly.express as px

from features import add_features

df = pd.read_csv("../data/raw/credit_clients.csv")
df.shape

In [ ]:
df.head()

## 1. Class balance — how rare is default?

In [ ]:
default_rate = df["default_next_month"].mean()
print(f"Default rate: {default_rate:.2%}")
px.histogram(df, x="default_next_month", title="Default next month (0 = no, 1 = yes)")

## 2. Feature engineering

Add utilization, payment ratio, and delinquency features.

In [ ]:
df_feat = add_features(df)
df_feat[["utilization", "payment_ratio_6m", "worst_delinquency_6m", "delinquency_trend"]].describe()

## 3. Does utilization predict default?

In [ ]:
px.box(df_feat, x="default_next_month", y="utilization",
       title="Utilization by default outcome", points=False)

## 4. Does delinquency history predict default?

In [ ]:
summary = (
    df_feat.groupby("worst_delinquency_6m")["default_next_month"]
    .mean()
    .reset_index(name="default_rate")
)
px.bar(summary, x="worst_delinquency_6m", y="default_rate",
       title="Default rate by worst repayment-status code (6m)")

## Next step

Run `python src/train_model.py` to train the logistic regression + XGBoost
models on these engineered features and produce the scored portfolio table
that `app.py` (the Streamlit dashboard) reads.